# Lab 7 — Gold: agregações e indicadores

## Objetivo

Este laboratório constrói a camada Gold a partir dos dados enriquecidos da Silver.

Serão produzidas duas tabelas agregadas. A primeira apresentará indicadores de risco por segmento de clientes. A segunda mostrará a evolução diária das transações e das fraudes.

As tabelas Gold são estruturadas para consumo direto por análises e ferramentas de Business Intelligence, evitando que o dashboard precise processar novamente todos os registros detalhados.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "silver").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "silver").exists():
            pasta_projeto = pasta_pai
            break

arquivo_silver = (
    pasta_projeto
    / "dados"
    / "silver"
    / "transactions_enriched.parquet"
)

pasta_gold = pasta_projeto / "dados" / "gold"
pasta_gold.mkdir(parents=True, exist_ok=True)

arquivo_gold_segmento_parquet = (
    pasta_gold / "fraud_risk.parquet"
)

arquivo_gold_segmento_csv = (
    pasta_gold / "fraud_risk.csv"
)

arquivo_gold_diario_parquet = (
    pasta_gold / "daily_metrics.parquet"
)

arquivo_gold_diario_csv = (
    pasta_gold / "daily_metrics.csv"
)

arquivo_banco = (
    pasta_projeto
    / "dia2_transformacao"
    / "lab07_gold"
    / "gold.duckdb"
)

assert arquivo_silver.exists(), "Arquivo Silver não encontrado."

print("Silver:", arquivo_silver)
print("Destino Gold:", pasta_gold)

Silver: C:\BigData\bigdata-curso-gabriel\dados\silver\transactions_enriched.parquet
Destino Gold: C:\BigData\bigdata-curso-gabriel\dados\gold


In [2]:
conexao = duckdb.connect(str(arquivo_banco))

caminho_silver = (
    arquivo_silver.as_posix().replace("'", "''")
)

conexao.execute(f"""
    CREATE OR REPLACE TABLE silver_transactions AS
    SELECT *
    FROM read_parquet('{caminho_silver}')
""")

resumo_silver = conexao.execute("""
    SELECT
        COUNT(*) AS transacoes,
        COUNT(DISTINCT transaction_id) AS ids_unicos,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes,
        SUM(amount) AS valor_total
    FROM silver_transactions
""").df()

resumo_silver

,transacoes,ids_unicos,fraudes,valor_total
0,100000,100000,1833.0,1.837908e+07


In [3]:
conexao.execute("""
    CREATE OR REPLACE TABLE gold_fraud_risk AS
    SELECT
        segment,
        COUNT(*) AS total_transacoes,
        COUNT(DISTINCT customer_id) AS total_clientes,
        ROUND(SUM(amount), 2) AS valor_total,
        ROUND(AVG(amount), 2) AS ticket_medio,

        SUM(
            CASE WHEN is_fraud THEN 1 ELSE 0 END
        ) AS qtd_fraudes,

        ROUND(
            100.0
            * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS taxa_fraude_pct,

        ROUND(
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END),
            2
        ) AS valor_em_risco

    FROM silver_transactions
    GROUP BY segment
""")

gold_segmento = conexao.execute("""
    SELECT *
    FROM gold_fraud_risk
    ORDER BY taxa_fraude_pct DESC
""").df()

gold_segmento

,segment,total_transacoes,total_clientes,valor_total,ticket_medio,qtd_fraudes,taxa_fraude_pct,valor_em_risco
0,High-Risk,9155,876,1664640.24,181.83,705.0,7.70,126451.07
1,Standard,29689,2665,5479399.85,184.56,655.0,2.21,106828.42
2,Premium,61156,5563,11235041.58,183.71,473.0,0.77,81308.44


In [4]:
conexao.execute("""
    CREATE OR REPLACE TABLE gold_daily_metrics AS

    WITH transacoes_com_data AS (
        SELECT
            CAST(transaction_timestamp AS DATE)
                AS transaction_date,
            transaction_id,
            customer_id,
            amount,
            is_fraud
        FROM silver_transactions
    )

    SELECT
        transaction_date,
        YEAR(transaction_date) AS year,
        MONTH(transaction_date) AS month,
        DAY(transaction_date) AS day,

        COUNT(*) AS transacoes,

        COUNT(DISTINCT customer_id) AS clientes_ativos,

        ROUND(SUM(amount), 2) AS valor_total,

        ROUND(AVG(amount), 2) AS ticket_medio,

        SUM(
            CASE WHEN is_fraud THEN 1 ELSE 0 END
        ) AS fraudes,

        ROUND(
            100.0
            * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS taxa_fraude_pct,

        ROUND(
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END),
            2
        ) AS valor_em_risco

    FROM transacoes_com_data
    GROUP BY
        transaction_date,
        YEAR(transaction_date),
        MONTH(transaction_date),
        DAY(transaction_date)
    ORDER BY transaction_date
""")

print("Gold diária criada.")

Gold diária criada.


In [5]:
resumo_gold_diaria = conexao.execute("""
    SELECT
        COUNT(*) AS quantidade_dias,
        MIN(transaction_date) AS primeira_data,
        MAX(transaction_date) AS ultima_data,
        SUM(transacoes) AS total_transacoes,
        SUM(fraudes) AS total_fraudes
    FROM gold_daily_metrics
""").df()

resumo_gold_diaria

,quantidade_dias,primeira_data,ultima_data,total_transacoes,total_fraudes
0,672,2023-01-01,2024-12-28,100000.0,1833.0


In [6]:
dias_maior_fraude = conexao.execute("""
    SELECT
        transaction_date,
        transacoes,
        fraudes,
        taxa_fraude_pct,
        valor_em_risco
    FROM gold_daily_metrics
    ORDER BY
        fraudes DESC,
        taxa_fraude_pct DESC,
        transaction_date
    LIMIT 10
""").df()

dias_maior_fraude

,transaction_date,transacoes,fraudes,taxa_fraude_pct,valor_em_risco
0,2023-04-27,353,16.0,4.53,2868.02
1,2023-12-22,335,13.0,3.88,2057.12
2,2024-05-23,350,13.0,3.71,2885.50
3,2023-05-21,359,13.0,3.62,1989.98
4,2024-05-25,310,12.0,3.87,2204.31
5,2024-08-26,307,11.0,3.58,1986.87
6,2024-12-28,309,11.0,3.56,3864.07
7,2024-08-28,314,11.0,3.50,1082.43
8,2024-01-26,332,11.0,3.31,1873.75
9,2024-12-24,363,11.0,3.03,1846.32


In [7]:
reconciliacao = conexao.execute("""
    SELECT
        indicador,
        silver,
        gold_segmento,
        gold_diaria,
        CASE
            WHEN silver = gold_segmento
             AND silver = gold_diaria
                THEN 'OK'
            ELSE 'DIVERGENTE'
        END AS validacao
    FROM (
        SELECT
            'Transações' AS indicador,

            (SELECT COUNT(*)
             FROM silver_transactions) AS silver,

            (SELECT SUM(total_transacoes)
             FROM gold_fraud_risk) AS gold_segmento,

            (SELECT SUM(transacoes)
             FROM gold_daily_metrics) AS gold_diaria

        UNION ALL

        SELECT
            'Fraudes' AS indicador,

            (SELECT SUM(
                CASE WHEN is_fraud THEN 1 ELSE 0 END
             )
             FROM silver_transactions) AS silver,

            (SELECT SUM(qtd_fraudes)
             FROM gold_fraud_risk) AS gold_segmento,

            (SELECT SUM(fraudes)
             FROM gold_daily_metrics) AS gold_diaria
    )
""").df()

reconciliacao

,indicador,silver,gold_segmento,gold_diaria,validacao
0,Transações,100000.0,100000.0,100000.0,OK
1,Fraudes,1833.0,1833.0,1833.0,OK


In [8]:
arquivos_saida = [
    arquivo_gold_segmento_parquet,
    arquivo_gold_segmento_csv,
    arquivo_gold_diario_parquet,
    arquivo_gold_diario_csv
]

for arquivo in arquivos_saida:
    if arquivo.exists():
        arquivo.unlink()

destino_segmento_parquet = (
    arquivo_gold_segmento_parquet.as_posix().replace("'", "''")
)

destino_segmento_csv = (
    arquivo_gold_segmento_csv.as_posix().replace("'", "''")
)

destino_diario_parquet = (
    arquivo_gold_diario_parquet.as_posix().replace("'", "''")
)

destino_diario_csv = (
    arquivo_gold_diario_csv.as_posix().replace("'", "''")
)

conexao.execute(f"""
    COPY gold_fraud_risk
    TO '{destino_segmento_parquet}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

conexao.execute(f"""
    COPY gold_fraud_risk
    TO '{destino_segmento_csv}'
    (
        FORMAT CSV,
        HEADER,
        DELIMITER ','
    )
""")

conexao.execute(f"""
    COPY gold_daily_metrics
    TO '{destino_diario_parquet}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

conexao.execute(f"""
    COPY gold_daily_metrics
    TO '{destino_diario_csv}'
    (
        FORMAT CSV,
        HEADER,
        DELIMITER ','
    )
""")

print("Arquivos Gold gravados com sucesso.")

Arquivos Gold gravados com sucesso.


In [9]:
validacao_arquivos_gold = conexao.execute(f"""
    SELECT
        'Gold por segmento' AS tabela,
        COUNT(*) AS linhas,
        SUM(total_transacoes) AS transacoes,
        SUM(qtd_fraudes) AS fraudes
    FROM read_parquet('{destino_segmento_parquet}')

    UNION ALL

    SELECT
        'Gold diária' AS tabela,
        COUNT(*) AS linhas,
        SUM(transacoes) AS transacoes,
        SUM(fraudes) AS fraudes
    FROM read_parquet('{destino_diario_parquet}')
""").df()

validacao_arquivos_gold

,tabela,linhas,transacoes,fraudes
0,Gold por segmento,3,100000.0,1833.0
1,Gold diária,672,100000.0,1833.0


In [10]:
indicadores_gerais = conexao.execute("""
    SELECT
        COUNT(*) AS total_transacoes,
        COUNT(DISTINCT customer_id) AS clientes_ativos,
        ROUND(SUM(amount), 2) AS valor_movimentado,
        ROUND(AVG(amount), 2) AS ticket_medio,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            AS total_fraudes,
        ROUND(
            100.0 * AVG(
                CASE WHEN is_fraud THEN 1 ELSE 0 END
            ),
            2
        ) AS taxa_fraude_pct,
        ROUND(
            SUM(
                CASE WHEN is_fraud THEN amount ELSE 0 END
            ),
            2
        ) AS valor_em_risco
    FROM silver_transactions
""").df()

indicadores_gerais

,total_transacoes,clientes_ativos,valor_movimentado,ticket_medio,total_fraudes,taxa_fraude_pct,valor_em_risco
0,100000,9104,18379081.67,183.79,1833.0,1.83,314587.93


In [11]:
conexao.close()

print("Conexão encerrada.")
print("Lab 7 executado com sucesso.")

Conexão encerrada.
Lab 7 executado com sucesso.


## Conclusão

A camada Gold foi construída em duas granularidades. A tabela `gold_fraud_risk` consolida os indicadores por segmento, permitindo comparar volume de transações, clientes, valores movimentados, quantidade de fraudes, taxa de fraude e valor financeiro em risco.

A tabela `gold_daily_metrics` apresenta os mesmos indicadores em periodicidade diária, possibilitando acompanhar a evolução temporal e identificar dias com maior ocorrência de fraude.

A reconciliação demonstrou que as duas tabelas preservam os totais da Silver: 100.000 transações e 1.833 fraudes. Dessa forma, os dados agregados permanecem consistentes com a camada detalhada.

As tabelas foram disponibilizadas em Parquet, para processamento analítico eficiente, e em CSV, para facilitar o consumo por ferramentas de visualização e Business Intelligence.